# ECG Report Image Classification with a CNN

A four-class image classification baseline. Run the code cells in order after following the README setup instructions. Training outputs are intentionally cleared.


In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import os
import cv2

# Dataset upload

## Optional Google Colab setup

For a dataset stored in Google Drive, run the following in a separate code cell:

```python
from google.colab import drive
drive.mount('/content/drive')
```

Then set `ECG_DATA_DIR` before running the configuration cell, or edit the fallback path below. Local Jupyter users can skip mounting Drive.


In [ ]:
# Run Jupyter from the repository root, or set an absolute dataset path.
data_path = os.environ.get("ECG_DATA_DIR", "data/ecg_report_images")
if not os.path.isdir(data_path):
    raise FileNotFoundError(
        f"Dataset directory not found: {data_path}. "
        "Set ECG_DATA_DIR or edit data_path to point to your class folders."
    )


## Training configuration

The baseline uses 240 × 240 RGB images and 25 epochs. `BATCH_SIZE` is declared in the supplied notebook but is not explicitly passed to `model.fit`; Keras uses its default.


In [ ]:
IMG_WIDTH, IMG_HEIGHT = 240, 240
BATCH_SIZE = 32
EPOCHS = 25

## Load and label images

The loader recognizes class folders using the phrases described in `data/README.md`. Each recognized folder should contain only readable image files.


In [ ]:
def load_image(path):
    img = cv2.imread(path)
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img / 255.0
    return img

images = []
labels = []


label_map = {'Abnormal_Heartbeat': 0, 'MI_history': 1, 'Myocardial_Infarction': 2, 'Normal':3}


for folder in os.listdir(data_path):
    folder_path = os.path.join(data_path, folder)

    # Determine the correct key for the label_map based on the folder name (case-insensitive check)
    label_key_for_folder = None
    folder_lower = folder.lower()
    if 'abnormal heartbeat' in folder_lower:
        label_key_for_folder = 'Abnormal_Heartbeat'
    elif 'history of mi' in folder_lower:
        label_key_for_folder = 'MI_history'
    elif 'myocardial infarction' in folder_lower:
        label_key_for_folder = 'Myocardial_Infarction'
    elif 'normal' in folder_lower:
        label_key_for_folder = 'Normal'

    if label_key_for_folder is not None:
        for file in os.listdir(folder_path):
            file_path = os.path.join(folder_path, file)
            img = load_image(file_path)
            images.append(img)
            labels.append(label_map[label_key_for_folder])
            # Removed print(labels) to avoid excessive output
    else:
        print(f"Warning: Folder '{folder}' does not match any defined label pattern. Skipping.")

images = np.array(images)
labels = np.array(labels)

In [ ]:
print(labels)

In [ ]:
from tensorflow.keras.utils import to_categorical

labels = to_categorical(labels)


In [ ]:
print(labels)

## Build the CNN


In [ ]:
model = keras.Sequential([
    keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_WIDTH, IMG_HEIGHT, 3)),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(4, activation='softmax')
])

In [ ]:
model.compile(optimizer='adam',
              loss= tf.keras.losses.CategoricalCrossentropy(),
              metrics=['accuracy'])

In [ ]:
print(model.summary())

## Split and train

This baseline uses an 80/20 split with `random_state=19`. The held-out split is used for both validation during training and the final evaluation. It is therefore a validation result, not an independent test result. The split is not stratified and no patient-level grouping is implemented.


In [ ]:
train_images, test_images, train_labels, test_labels = train_test_split(images, labels, test_size=0.2, random_state=19)

In [ ]:
print(labels.shape)
print(images.shape)

In [ ]:
history = model.fit(
    train_images, train_labels,
    validation_data=(test_images, test_labels),
    epochs = EPOCHS
)

## Evaluate the held-out split

The original variable names and printed “Test accuracy” label are retained. Interpret this value as accuracy on the reused validation split.


In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels)
print(f'Test accuracy: {test_acc:.2f}')

## Export model artifacts

The baseline writes `model.h5` and then `model.h5.zip` into the working directory. These generated files are excluded from Git by `.gitignore`.


In [ ]:
model.save('model.h5')

In [ ]:
import zipfile

with zipfile.ZipFile('model.h5.zip', 'w') as zip_file:
    zip_file.write('model.h5')